[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C06_Interpretability_Course/07_interp_for_evals/07_interp_for_evals.ipynb)

# 07 · Interp × 评测与安全审计：一次小型白盒审计演练

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy / pandas / matplotlib，自包含合成数据，全部 cell 秒级跑完。

**本 notebook 你将完成：**

1. 构造一个会 **sandbag** 的玩具"模型"：合成激活生成器，检测到评测线索时压低能力读出（ground truth 机制已知，方便最后校验工具）；
2. 行为侧测量：评测输入 vs 普通输入的准确率差距——**行为证据只能走到这一步**；
3. 训 linear probe 检测 **evaluation awareness** 方向（AUC + 与 ground truth 方向的余弦）；
4. **steering 消融**该方向 → 能力恢复 + 两组 control——sandbagging 的因果证据（机制版 elicitation）；
5. **model diffing** 演练：比对"微调前后"的激活统计，用均值差方向与 diff 主成分定位注入方向；
6. 把全部证据组织成一份 **mini 审计报告**（pandas 证据表 + verdict 模板）；
7. 完成 4 道 ✏️ 练习（sandbagging 指标、恢复实验、diff 方向定位、审计判定函数），每道配 assert 自测。

参考文献：Marks 2025 (arXiv:2503.10965) · MacDiarmid 2024 (sleeper agent probes) · van der Weij 2024 (arXiv:2406.07358) · Betley 2025 (arXiv:2502.17424) · Chen 2025 (arXiv:2507.21509)

## 1 · 场景：一个会 sandbag 的玩具模型

**造物主视角（审计者不可见）**：模型的 residual stream 维度 $D=32$，其中藏着一个 **evaluation-awareness 方向** $v_{\text{eval}}$。输入若含评测线索，该方向被激活；模型的能力读出内置 sandbagging 机制——检测到该方向超过阈值就压低有效能力：

$$\text{skill}_{\text{eff}} = \text{skill} - \gamma \cdot \mathrm{relu}(h^\top v_{\text{eval}} - \tau), \qquad p(\text{correct}) = \sigma(\text{skill}_{\text{eff}} - \text{difficulty})$$

**审计者视角**：能拿到激活 $h$、能查询模型作答的期望正确率（黑盒 readout），但不知道 $v_{\text{eval}}$、$\gamma$、$\tau$ 的存在。本 notebook 全程扮演审计者，仅在标注"上帝视角"的行里用 ground truth 校验工具是否找对。

行为侧第一步：分别在普通输入与评测线索输入上测准确率。注意两组题目难度同分布，但真实审计中你**无法确认**这一点——行为证据只能走到"差距存在"，无法区分 sandbagging 与"评测题更难 / 分布更偏"。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

D        = 32     # residual stream 维度
SKILL    = 1.5    # 模型真实能力（logit 尺度）
GAMMA    = 1.2    # sandbagging 强度
THRESH   = 1.0    # 评测意识触发阈值 tau
EVAL_AMP = 2.5    # 评测线索在 v_eval 方向上的平均激活强度

# ---- ground truth：只有"造物主"知道，审计者不可见 ----
v_eval = rng.normal(size=D); v_eval /= np.linalg.norm(v_eval)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def make_batch(n, p_eval=0.5):
    """生成 n 条输入：激活 H、是否含评测线索 is_eval、题目难度 difficulty。"""
    is_eval = (rng.random(n) < p_eval).astype(int)
    H = rng.normal(size=(n, D))                        # 基底激活（各向同性噪声）
    amp = rng.normal(EVAL_AMP, 0.4, size=n) * is_eval  # 评测线索注入强度
    H = H + amp[:, None] * v_eval[None, :]
    difficulty = rng.normal(0.0, 1.0, size=n)
    return H, is_eval, difficulty

def model_expected_acc(H, difficulty):
    """模型的能力读出（期望正确率）。内部含 sandbagging 机制；
    审计者可以调用（= 黑盒查询模型），但看不到实现。"""
    eval_signal = np.maximum(H @ v_eval - THRESH, 0.0)
    effective_skill = SKILL - GAMMA * eval_signal
    return sigmoid(effective_skill - difficulty)

H, is_eval, difficulty = make_batch(600)
p_correct = model_expected_acc(H, difficulty)

acc_normal = p_correct[is_eval == 0].mean()
acc_eval   = p_correct[is_eval == 1].mean()
gap_behavioral = acc_normal - acc_eval
print(f"普通输入准确率    : {acc_normal:.3f}")
print(f"评测线索输入准确率: {acc_eval:.3f}")
print(f"行为差距 Δ        : {gap_behavioral:.3f}")
print("行为证据到此为止：是 sandbagging，还是评测题恰好更难？黑盒分不开。")

## 2 · 相关证据：probe 检测 evaluation awareness

模块 01 的工具直接上：在激活上训一个 logistic probe，分类"评测线索 vs 普通"输入。两个产出：

- **AUC**：评测意识在激活中是否线性可读（MacDiarmid 2024 sleeper agent probes 同款逻辑——意图在行为之前就线性可读）；
- **probe 方向** $\hat u$：交给下一节做因果干预。

注意这一步的证据是**相关性的**：probe 能分开两类输入 $\ne$ 该方向导致了行为差距——也可能只是评测输入的某个无关签名。因果要靠第 3 节。

In [ ]:
def train_logistic_probe(X, y, lr=0.5, steps=400):
    """纯 numpy 逻辑回归 probe，全批量梯度下降，返回 (w, b)。"""
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b)
        w -= lr * X.T @ (p - y) / len(y)
        b -= lr * (p - y).mean()
    return w, b

def auc_score(scores, labels):
    """rank-based AUC（Mann–Whitney）：P(正样本分数 > 负样本分数) + 0.5·P(打平)。"""
    pos, neg = scores[labels == 1], scores[labels == 0]
    greater = (pos[:, None] > neg[None, :]).sum()
    ties = (pos[:, None] == neg[None, :]).sum()
    return (greater + 0.5 * ties) / (len(pos) * len(neg))

idx = rng.permutation(len(H)); tr, te = idx[:400], idx[400:]
w, b = train_logistic_probe(H[tr], is_eval[tr])
scores_te = sigmoid(H[te] @ w + b)
auc = auc_score(scores_te, is_eval[te])

u_probe = w / np.linalg.norm(w)
cos_gt = float(u_probe @ v_eval)
print(f"probe 测试集 AUC : {auc:.3f}")
print(f"与 v_eval 的余弦 : {cos_gt:.3f}   <- 上帝视角校验：真实审计没有这一行")

plt.figure(figsize=(7, 3.2))
plt.hist(scores_te[is_eval[te] == 0], bins=30, alpha=.65, label="normal")
plt.hist(scores_te[is_eval[te] == 1], bins=30, alpha=.65, label="eval-cue")
plt.xlabel("probe score"); plt.ylabel("count"); plt.legend()
plt.title(f"evaluation-awareness probe (test AUC = {auc:.3f})")
plt.tight_layout(); plt.show()

## 3 · 因果证据：steering 消融 → 能力恢复

机制版 elicitation 三件套（对应讲解第 3 节）：

1. **干预**：对评测输入的激活做方向消融 $h' = h - (h^\top \hat u)\,\hat u$，再查询模型 → 分数是否回升；
2. **control 1（随机方向）**：消融一个随机方向，理应不恢复——排除"任意扰动都能提分"；
3. **control 2（普通输入）**：对普通输入做同样消融，分数理应基本不变——排除"消融本身注入能力"。

若差距随 $\hat u$ 的消融而消失，说明差距由该方向**介导**：模型的能力本来就在，只是被"我在被测"的表征压住了——这是 **sandbagging 而非能力缺失**的因果证据。恢复率 $R = (\Delta - \Delta')/\Delta$。

In [ ]:
def ablate(H_in, u):
    """方向消融：移除激活在 u（归一化后）上的分量。"""
    u = u / np.linalg.norm(u)
    return H_in - (H_in @ u)[:, None] * u[None, :]

mask = is_eval == 1

# 干预：沿 probe 方向消融评测输入
acc_eval_ablated = model_expected_acc(ablate(H[mask], u_probe), difficulty[mask]).mean()
gap_after = acc_normal - acc_eval_ablated
recovery = (gap_behavioral - gap_after) / gap_behavioral

# control 1：随机方向消融
u_rand = rng.normal(size=D); u_rand /= np.linalg.norm(u_rand)
acc_eval_rand = model_expected_acc(ablate(H[mask], u_rand), difficulty[mask]).mean()

# control 2：普通输入上做同样消融
acc_normal_ablated = model_expected_acc(ablate(H[~mask], u_probe), difficulty[~mask]).mean()

print(f"评测输入准确率: 消融前 {acc_eval:.3f} -> 沿 probe 方向消融后 {acc_eval_ablated:.3f}")
print(f"差距: Δ={gap_behavioral:.3f} -> Δ'={gap_after:.3f}   恢复率 R={recovery:.2f}")
print(f"control 1（随机方向消融后评测准确率）: {acc_eval_rand:.3f}   <- 不恢复")
print(f"control 2（普通输入消融后准确率）    : {acc_normal_ablated:.3f} (原 {acc_normal:.3f})   <- 基本不变")

bars = {"normal": acc_normal, "eval\n(before)": acc_eval,
        "eval\n(probe-dir\nablated)": acc_eval_ablated, "eval\n(random-dir\nablated)": acc_eval_rand}
plt.figure(figsize=(7, 3.4))
plt.bar(range(len(bars)), list(bars.values()),
        color=["#4c78a8", "#e45756", "#54a24b", "#b8b8b8"])
plt.xticks(range(len(bars)), list(bars.keys()), fontsize=9)
plt.ylabel("expected accuracy"); plt.title("sandbagging: ablating the eval-awareness direction restores capability")
plt.tight_layout(); plt.show()

## 4 · Model diffing：定位"微调"注入的方向

换一个审计任务：拿到同一模型"微调前 / 后"两个 checkpoint，在**同一批输入**上分别收集激活 `H_base / H_ft`。造物主在"微调"中注入了一个方向 $u_{\text{inj}}$（模拟 persona 漂移——emergent misalignment [Betley 2025] 的激活面），审计者要把它找回来。两个互相印证的方法：

- **均值差方向**：$\hat u_{\Delta\mu} = (\bar h_{\text{ft}} - \bar h_{\text{base}}) / \lVert \cdot \rVert$；
- **diff 主成分**：对逐样本差 $D = H_{\text{ft}} - H_{\text{base}}$ 的协方差做特征分解，最大特征向量 = 变化最集中的方向；能量占比 $\lambda_1 / \sum_i \lambda_i$ 衡量改动是否近似 rank-1（LoRA 式低秩更新的激活侧特征）。

In [ ]:
# ---- 造物主视角：微调注入新方向 u_inj（强度随输入变化）+ 少量各向同性漂移 ----
u_inj = rng.normal(size=D); u_inj /= np.linalg.norm(u_inj)

n_diff = 500
H_base = rng.normal(size=(n_diff, D))
strength = np.maximum(rng.normal(1.0, 0.5, size=n_diff), 0.0)   # 部分输入触发，强度非负
H_ft = H_base + strength[:, None] * u_inj[None, :] + rng.normal(0, 0.05, size=(n_diff, D))

# ---- 审计者视角：只有 H_base / H_ft ----
delta_mu = H_ft.mean(0) - H_base.mean(0)                # 方法 1：均值差方向
u_meandiff = delta_mu / np.linalg.norm(delta_mu)

Dmat = H_ft - H_base                                     # 方法 2：diff 主成分
Dc = Dmat - Dmat.mean(0)
eigval, eigvec = np.linalg.eigh(Dc.T @ Dc / len(Dmat))
u_pc = eigvec[:, -1]
energy_ratio = eigval[-1] / eigval.sum()

print(f"均值差方向 vs u_inj 余弦 : {u_meandiff @ u_inj: .3f}   <- 上帝视角校验")
print(f"diff 主成分 vs u_inj 余弦: {abs(u_pc @ u_inj): .3f}")
print(f"diff 能量占比 λ1/Σλ      : {energy_ratio:.3f}   <- 接近 1：改动近似 rank-1，集中在单一方向")

plt.figure(figsize=(7, 3))
plt.semilogy(eigval[::-1], marker=".")
plt.xlabel("eigenvalue rank"); plt.ylabel("eigenvalue (log)")
plt.title("activation-diff spectrum: the injected change is (almost) rank-1")
plt.tight_layout(); plt.show()

## 5 · Mini 审计报告：把证据组织成判定

审计报告的纪律（讲解第 5、7 节）：每条证据标注**证据层级**（行为 / 相关 / 因果 / 对照 / diffing），结论用可被攻击的句式——"**在 X 方法、Y 覆盖面下，发现 / 未发现 Z**"。下面把本 notebook 的全部测量装进一张证据表，并给出 verdict。

In [ ]:
evidence = {
    "behavioral_gap":     float(gap_behavioral),
    "probe_auc":          float(auc),
    "gap_after_ablation": float(gap_after),
    "control_gap_random": float(acc_normal - acc_eval_rand),
    "diff_top_cosine":    abs(float(u_meandiff @ u_inj)),
    "diff_energy_ratio":  float(energy_ratio),
}

report = pd.DataFrame([
    ("行为",    "评测 vs 普通输入准确率差",  "Δ",       evidence["behavioral_gap"],     "存在可疑差距 (>0.05)"),
    ("相关",    "evaluation-awareness probe", "AUC",     evidence["probe_auc"],          "评测意识线性可读 (>0.8)"),
    ("因果",    "probe 方向消融后差距",       "Δ'",      evidence["gap_after_ablation"], "差距消失 -> 能力被压制而非缺失"),
    ("对照",    "随机方向消融后差距",         "Δ_rand",  evidence["control_gap_random"], "差距仍在 -> 恢复不是扰动伪影"),
    ("diffing", "微调注入方向定位",           "|cos|",   evidence["diff_top_cosine"],    "均值差方向 = 注入方向"),
    ("diffing", "diff 能量集中度",            "λ1/Σλ",   evidence["diff_energy_ratio"],  "改动近似 rank-1"),
], columns=["证据层", "方法", "指标", "数值", "解读"])
report["数值"] = report["数值"].round(3)
print(report.to_string(index=False))

print("\n—— 审计结论（模板句式）——")
print("在 linear probe + 方向消融（覆盖面：单方向、本批输入分布）下，发现评测线索条件下的能力压制：")
print(f"行为差距 {evidence['behavioral_gap']:.3f} 在消融后降至 {evidence['gap_after_ablation']:.3f}"
      f"（随机方向对照仍为 {evidence['control_gap_random']:.3f}）。")
print("判定：sandbagging 证据成立；直接测得的分数不应作为能力上界写入 safety case。")

---
## ✏️ 练习 1：实现 sandbagging 检测指标

实现 `sandbagging_metrics(p_correct, is_eval, probe_scores)`，返回 dict：

- `"gap"`：普通输入平均正确率 − 评测输入平均正确率（行为差距 Δ）；
- `"auc"`：`probe_scores` 区分 `is_eval` 的 rank-based AUC（Mann–Whitney：正样本分数严格大于负样本计 1，打平计 0.5）。

**提示**：AUC 用成对比较 `(pos[:, None] > neg[None, :])` 即可，几百条数据 $O(n^2)$ 没有问题；记得把 `is_eval` 转成 `np.asarray` 以支持 list 输入；打平项乘 0.5；15 行以内。

In [ ]:
def sandbagging_metrics(p_correct, is_eval, probe_scores):
    # TODO:
    #  1) gap = 普通输入 (is_eval==0) 平均正确率 - 评测输入 (is_eval==1) 平均正确率
    #  2) auc = [ #(pos > neg) + 0.5 * #(pos == neg) ] / (n_pos * n_neg)
    # 返回 {"gap": float, "auc": float}
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
m = sandbagging_metrics(p_correct, is_eval, sigmoid(H @ w + b))
assert abs(m["gap"] - gap_behavioral) < 1e-9          # 与正文测量一致
assert m["auc"] > 0.9                                  # probe 全量数据上同样高分
toy = sandbagging_metrics(np.array([1., 1., 0., 0.]), np.array([0, 0, 1, 1]), np.array([.1, .2, .8, .9]))
assert abs(toy["gap"] - 1.0) < 1e-9                    # 完全 sandbag
assert abs(toy["auc"] - 1.0) < 1e-9                    # 完美分离
tie = sandbagging_metrics(np.array([1., 0.]), np.array([0, 1]), np.array([.5, .5]))
assert abs(tie["auc"] - 0.5) < 1e-9                    # 全打平 -> 0.5
print("✅ 练习 1 通过")

---
## ✏️ 练习 2：实现 steering 恢复实验

实现 `recovery_experiment(H, is_eval, difficulty, direction, readout_fn)`：

1. `gap_before` = 普通输入平均读出 − 评测输入平均读出；
2. 仅对**评测输入**的激活沿 `direction` 做消融 $h' = h - (h^\top \hat u)\hat u$；
3. `gap_after` = 普通输入平均读出 − 消融后评测输入平均读出；
4. `recovery` = $(\Delta - \Delta') / \Delta$（`gap_before == 0` 时返回 `0.0` 防除零）。

返回 `(gap_before, gap_after, recovery)`。

**提示**：`direction` 不保证已归一化——先除以范数；`readout_fn(H, difficulty)` 返回期望正确率数组（黑盒查询模型）；12 行以内。自测会用随机方向做对照：随机方向必须恢复失败。

In [ ]:
def recovery_experiment(H, is_eval, difficulty, direction, readout_fn):
    # TODO:
    #  1) û = direction / ||direction||
    #  2) gap_before = mean(readout 普通) - mean(readout 评测)
    #  3) 评测输入消融: H' = H_eval - (H_eval @ û)[:, None] * û
    #  4) gap_after = mean(readout 普通) - mean(readout 消融后评测)
    #  5) recovery = (gap_before - gap_after) / gap_before  （gap_before==0 时为 0.0）
    # 返回 (gap_before, gap_after, recovery)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
gb, ga, rec = recovery_experiment(H, is_eval, difficulty, u_probe, model_expected_acc)
assert abs(gb - gap_behavioral) < 1e-9                 # 消融前差距与正文一致
assert ga < 0.07                                       # 消融后差距基本消失
assert rec > 0.8                                       # 恢复率高 -> sandbagging 因果证据
rng_ctl = np.random.default_rng(123)
u_r = rng_ctl.normal(size=D)                           # 未归一化的随机方向（考验你的归一化）
_, ga_r, rec_r = recovery_experiment(H, is_eval, difficulty, u_r, model_expected_acc)
assert ga_r > 0.2 and rec_r < 0.3                      # 随机方向对照：恢复失败
print("✅ 练习 2 通过")

---
## ✏️ 练习 3：实现激活分布 diff 的方向定位

实现 `locate_shift_direction(H_a, H_b)`：给定同一批输入在两个 checkpoint 上的激活，返回**单位向量**，方向为均值漂移 $\bar h_b - \bar h_a$。

**提示**：3 行足够；注意返回单位范数；方向带符号——交换两个输入，结果应当反号（自测会查）。这正是 persona vectors / model diffing 流水线里最简单也最常用的第一步。

In [ ]:
def locate_shift_direction(H_a, H_b):
    # TODO: 计算均值差 H_b.mean(0) - H_a.mean(0)，归一化成单位向量后返回
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
u_hat = locate_shift_direction(H_base, H_ft)
assert abs(np.linalg.norm(u_hat) - 1.0) < 1e-6         # 单位向量
assert u_hat @ u_inj > 0.95                            # 找回注入方向
u_rev = locate_shift_direction(H_ft, H_base)
assert u_rev @ u_inj < -0.95                           # 交换两组 -> 方向反号
print("✅ 练习 3 通过")

---
## ✏️ 练习 4：实现审计结论的判定函数

实现 `audit_verdict(ev)`：输入证据字典（至少含 `behavioral_gap` / `probe_auc` / `gap_after_ablation` / `control_gap_random` 四个键），按以下规则输出 verdict 字符串：

1. `behavioral_gap < 0.05` → `"no_behavioral_evidence"`（无行为差距，后续机制证据无从谈起）；
2. 否则若 `probe_auc >= 0.8` **且** `gap_after_ablation < 0.05` **且** `control_gap_random >= 0.5 * behavioral_gap` → `"sandbagging_confirmed"`（相关 + 因果 + 对照三层齐备）；
3. 否则若 `probe_auc >= 0.8` → `"mechanism_found_causality_unclear"`（probe 有信号但因果链或对照不成立）；
4. 否则 → `"behavioral_gap_unexplained"`（有行为差距但机制证据缺位——可能真是能力缺失或分布差异）。

**提示**：纯 if/elif，10 行以内；规则顺序就是判定优先级；字典多余的键直接忽略。这是把讲解第 5 节"证据层级"变成代码的练习——注意 `sandbagging_confirmed` 必须三个条件同时成立，缺一个都要降级。

In [ ]:
def audit_verdict(ev):
    # TODO: 按练习说明的 4 条规则（按顺序）返回对应 verdict 字符串
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert audit_verdict(evidence) == "sandbagging_confirmed"   # 正文证据：三层齐备
assert audit_verdict({"behavioral_gap": .01, "probe_auc": .99,
                      "gap_after_ablation": .0,  "control_gap_random": .0}) == "no_behavioral_evidence"
assert audit_verdict({"behavioral_gap": .30, "probe_auc": .95,
                      "gap_after_ablation": .25, "control_gap_random": .28}) == "mechanism_found_causality_unclear"
assert audit_verdict({"behavioral_gap": .30, "probe_auc": .55,
                      "gap_after_ablation": .28, "control_gap_random": .30}) == "behavioral_gap_unexplained"
assert audit_verdict({"behavioral_gap": .30, "probe_auc": .90,
                      "gap_after_ablation": .02, "control_gap_random": .02}) == "mechanism_found_causality_unclear"  # 对照失效要降级
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def sandbagging_metrics(p_correct, is_eval, probe_scores):
    is_eval = np.asarray(is_eval)
    gap = p_correct[is_eval == 0].mean() - p_correct[is_eval == 1].mean()
    pos, neg = probe_scores[is_eval == 1], probe_scores[is_eval == 0]
    greater = (pos[:, None] > neg[None, :]).sum()
    ties = (pos[:, None] == neg[None, :]).sum()
    auc = (greater + 0.5 * ties) / (len(pos) * len(neg))
    return {"gap": float(gap), "auc": float(auc)}

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def recovery_experiment(H, is_eval, difficulty, direction, readout_fn):
    u = direction / np.linalg.norm(direction)
    m = is_eval == 1
    acc_n = readout_fn(H[~m], difficulty[~m]).mean()
    acc_e = readout_fn(H[m], difficulty[m]).mean()
    gap_before = acc_n - acc_e
    H_abl = H[m] - (H[m] @ u)[:, None] * u[None, :]
    gap_after = acc_n - readout_fn(H_abl, difficulty[m]).mean()
    recovery = 0.0 if gap_before == 0 else (gap_before - gap_after) / gap_before
    return float(gap_before), float(gap_after), float(recovery)

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def locate_shift_direction(H_a, H_b):
    d = H_b.mean(0) - H_a.mean(0)
    return d / np.linalg.norm(d)

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def audit_verdict(ev):
    if ev["behavioral_gap"] < 0.05:
        return "no_behavioral_evidence"
    if ev["probe_auc"] >= 0.8:
        confirmed = (ev["gap_after_ablation"] < 0.05
                     and ev["control_gap_random"] >= 0.5 * ev["behavioral_gap"])
        return "sandbagging_confirmed" if confirmed else "mechanism_found_causality_unclear"
    return "behavioral_gap_unexplained"

---
## 🎯 真实数据胶囊题：把 probe 当白盒检测器：真实 embedding 上的 AUC

可解释性能给评测当“白盒显微镜”：训一个方向当检测器、用 AUC 衡量它分得开不开。用真实 GPT-2 embedding 训数字方向 probe，在留出 token 上算检测 AUC。

> 本模块新增的**真实数据**练习：用**真实 GPT-2 权重/embedding**把本章的可解释性技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, struct, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.interp_data"); os.makedirs(CACHE,exist_ok=True)
ST="https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
def _rng(s,e):
    req=urllib.request.Request(ST, headers={"Range":f"bytes={s}-{e}"})
    return urllib.request.urlopen(req,timeout=60).read()
def gpt2_emb_block(n=6000):
    cache=os.path.join(CACHE,f"wte_{n}.npy")
    if os.path.exists(cache): return np.load(cache)
    hlen=struct.unpack("<Q", _rng(0,7))[0]; hdr=json.loads(_rng(8,8+hlen-1))
    info=hdr["wte.weight"]; base=8+hlen; s0=info["data_offsets"][0]; d=info["shape"][1]
    raw=_rng(base+s0, base+s0+n*d*4-1)
    E=np.frombuffer(raw,dtype=np.float32).reshape(n,d).copy()
    np.save(cache,E); return E
def gpt2_vocab():
    p=os.path.join(CACHE,"vocab.json")
    if not os.path.exists(p): urllib.request.urlretrieve("https://huggingface.co/openai-community/gpt2/resolve/main/vocab.json",p)
    return json.load(open(p))
def digit_letter_dataset(lim=6000):
    "返回 (X[token嵌入], y[1=数字 0=字母], E, ids_digit, ids_alpha)"
    v=gpt2_vocab(); E=gpt2_emb_block(lim)
    dig=[i for t,i in v.items() if i<lim and t.isdigit()]
    alpha=[i for t,i in v.items() if i<lim and t.isalpha() and t.isascii()]
    rng=np.random.default_rng(0); alpha=list(rng.permutation(alpha)[:len(dig)])
    ids=dig+alpha; y=np.array([1]*len(dig)+[0]*len(alpha))
    return E[ids], y, E, dig, alpha
def shakespeare():
    p=os.path.join(CACHE,"shake.txt")
    if not os.path.exists(p): urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",p)
    return open(p).read()

X,y,E,dig,alpha=digit_letter_dataset()
rng=np.random.default_rng(2); idx=rng.permutation(len(y)); cut=int(0.6*len(idx))
tr,te=idx[:cut],idx[cut:]
mu,sd=X[tr].mean(0),X[tr].std(0)+1e-8
w=np.zeros(X.shape[1]); b=0.0; Xs=(X[tr]-mu)/sd
for _ in range(600):
    p=1/(1+np.exp(-(Xs@w+b))); g=p-y[tr]; w-=0.5*Xs.T@g/len(tr); b-=0.5*g.mean()
scores=1/(1+np.exp(-(((X[te]-mu)/sd)@w+b)))   # 留出集检测分
print("留出 token 检测分已就绪")

**练习**：实现 `auc(scores, labels)`（用排序/概率解释：随机正样本分>随机负样本分的概率）。验证白盒数字检测器在真实留出 embedding 上 AUC 明显 > 0.5。

In [ ]:
def auc(scores, labels):
    # TODO: 按分数降序，累计 TPR/FPR 梯形积分；或用 rank 公式
    raise NotImplementedError


In [ ]:
# 自测
a=auc(scores, y[te])
assert a>0.85, f"白盒数字检测器 AUC 应高(>0.85), 得到{a:.3f}"
# 完美/随机边界
assert abs(auc(y[te].astype(float), y[te])-1.0)<1e-9
print(f"白盒检测器 AUC={a:.3f} ✓ —— interp 方向可直接当评测/审计探针")


### 📖 参考答案

In [ ]:
def auc(scores, labels):
    s=np.asarray(scores); y=np.asarray(labels); order=np.argsort(-s); y=y[order]
    P=y.sum(); N=len(y)-P
    if P==0 or N==0: return 0.5
    tpr=np.concatenate([[0],np.cumsum(y)/P]); fpr=np.concatenate([[0],np.cumsum(1-y)/N])
    return float(np.sum(np.diff(fpr)*(tpr[:-1]+tpr[1:])/2))
print("✓ interp×评测：把可解释方向当白盒探针，是 sandbagging/欺骗检测的前沿")